# 📊 SQL Analysis & Views Showcase (SQLite3 & Python)

Notebook ini mendemonstrasikan secara lengkap eksekusi **15 query analitik SQL** dan pembuatan **4 Database Views** yang digunakan untuk menganalisis performa produksi, downtime mesin, dan tingkat kecacatan produk pada PT Voltec Indonesia.

### 🛠️ Alur Kerja:
1. Membuat database SQLite lokal (`production.db`).
2. Membaca skema DDL dari `schema.sql` dan membangun struktur tabel relasional.
3. Meng-import data bersih dari folder `data/cleaned/` (CSV) menggunakan Pandas ke dalam tabel database.
4. Meregistrasi fungsi statistik kustom (**STDDEV**) ke SQLite menggunakan Python.
5. Membuat Database Views (`views.sql`) untuk ringkasan laporan.
6. Menjalankan seluruh 15 query analisis secara berurutan dan menampilkan hasilnya menggunakan Pandas DataFrame.

## 1. Setup & Inisialisasi Database

In [1]:
import sqlite3
import pandas as pd
import numpy as np
import math
import os

# Tentukan path database di folder sql
db_path = "production.db"
if os.path.exists(db_path):
    os.remove(db_path)  # Reset DB lama agar bersih

# Sambungkan ke SQLite
conn = sqlite3.connect(db_path)
print("Koneksi ke database SQLite berhasil dibuat.")

Koneksi ke database SQLite berhasil dibuat.


## 2. Registrasi Custom Aggregate Function (STDDEV)
SQLite3 secara bawaan tidak menyediakan fungsi deviasi standar (`STDDEV`). Kita mendaftarkan fungsi kustom ke engine SQLite menggunakan API Python.

In [2]:
class StdDev:
    def __init__(self):
        self.M = 0.0
        self.S = 0.0
        self.k = 1

    def step(self, value):
        if value is None:
            return
        x = float(value)
        if self.k == 1:
            self.M = x
            self.S = 0.0
        else: 
            old_M = self.M
            self.M += (x - self.M) / self.k
            self.S += (x - old_M) * (x - self.M)
        self.k += 1

    def finalize(self): 
        if self.k <= 2:
            return 0.0
        return math.sqrt(self.S / (self.k - 2))  # Sampel Standar Deviasi

conn.create_aggregate("STDDEV", 1, StdDev)
print("Fungsi kustom STDDEV berhasil diregistrasikan ke database SQLite.")

Fungsi kustom STDDEV berhasil diregistrasikan ke database SQLite.


## 3. Eksekusi Skema DDL (`schema.sql`)
Kita membaca skema tabel relasional (Star Schema) dan mengeksekusinya di SQLite.

In [3]:
# Aktifkan dukungan Foreign Keys di SQLite
conn.execute("PRAGMA foreign_keys = ON;")

with open("schema.sql", "r", encoding="utf-8") as f:
    schema_sql = f.read()

# Penyesuaian tipe data DECIMAL untuk kompatibilitas SQLite
schema_sql = schema_sql.replace("DECIMAL(10,2)", "NUMERIC")
schema_sql = schema_sql.replace("DECIMAL(6,1)", "NUMERIC")

# Eksekusi seluruh script DDL
conn.executescript(schema_sql)
conn.commit()
print("Tabel dimensi dan fakta berhasil dibuat di SQLite.")

Tabel dimensi dan fakta berhasil dibuat di SQLite.


## 4. Import Data CSV ke Database
Membaca data bersih hasil wrangling yang disimpan di folder `data/cleaned/` lalu memuatnya ke tabel relasional database.

In [4]:
# Import Tabel Dimensi
pd.read_csv("../data/cleaned/line_dim_cleaned.csv").to_sql("line_dim", conn, if_exists="append", index=False)
pd.read_csv("../data/cleaned/machine_dim_cleaned.csv").to_sql("machine_dim", conn, if_exists="append", index=False)
pd.read_csv("../data/cleaned/shift_dim_cleaned.csv").to_sql("shift_dim", conn, if_exists="append", index=False)
pd.read_csv("../data/cleaned/product_dim_cleaned.csv").to_sql("product_dim", conn, if_exists="append", index=False)

# Import Tabel Fakta
pd.read_csv("../data/cleaned/production_fact_cleaned.csv").to_sql("production_fact", conn, if_exists="append", index=False)

print("Seluruh data CSV berhasil di-import ke database SQLite.")

Seluruh data CSV berhasil di-import ke database SQLite.


In [5]:
tables = ["line_dim", "machine_dim", "shift_dim", "product_dim", "production_fact"]
for t in tables:
    row_count = pd.read_sql_query(f"SELECT COUNT(*) AS total_rows FROM {t}", conn)["total_rows"][0]
    print(f"Tabel: {t:<17} | Jumlah Baris: {row_count}")

Tabel: line_dim          | Jumlah Baris: 5
Tabel: machine_dim       | Jumlah Baris: 15
Tabel: shift_dim         | Jumlah Baris: 3
Tabel: product_dim       | Jumlah Baris: 5
Tabel: production_fact   | Jumlah Baris: 24718


## 5. Membuat Database Views (`views.sql`)
Views digunakan untuk membungkus query analisis yang kompleks agar siap dikonsumsi langsung oleh Power BI.

In [6]:
with open("views.sql", "r", encoding="utf-8") as f:
    views_sql = f.read()

# Penyesuaian sintaks PostgreSQL EXTRACT ke strftime SQLite
views_sql = views_sql.replace(
    "EXTRACT(MONTH FROM production_date)",
    "CAST(strftime('%m', production_date) AS INTEGER)"
)
views_sql = views_sql.replace(
    "EXTRACT(MONTH FROM pf.production_date)",
    "CAST(strftime('%m', pf.production_date) AS INTEGER)"
)

# Eksekusi pembuatan views
conn.executescript(views_sql)
conn.commit()
print("Database Views (v_monthly_executive_kpi, dll) berhasil dibuat di SQLite.")

Database Views (v_monthly_executive_kpi, dll) berhasil dibuat di SQLite.


## 6. Uji Coba 15 Query Analisis (Showcase)
Berikut adalah 15 query analisis lengkap dari `analysis.sql` yang disesuaikan dengan engine SQLite3.

### Query 1: Total Output & Efisiensi Keseluruhan
**Deskripsi**: Mengukur total record, target, output, defect, persentase pencapaian (achievement rate), tingkat defect (defect rate), dan rata-rata downtime secara global.

In [7]:
query_1 = """
SELECT
    COUNT(*) AS total_records,
    SUM(output_qty) AS total_output,
    SUM(target_qty) AS total_target,
    SUM(defect_qty) AS total_defect,
    ROUND(SUM(output_qty) * 100.0 / SUM(target_qty), 1) AS achievement_pct,
    ROUND(SUM(defect_qty) * 100.0 / SUM(output_qty), 2) AS defect_rate_pct,
    ROUND(AVG(downtime_minutes), 1) AS avg_downtime
FROM production_fact;
"""
pd.read_sql_query(query_1, conn)

,total_records,total_output,total_target,total_defect,achievement_pct,defect_rate_pct,avg_downtime
0,24718,961992,1052628,31956,91.4,3.32,21.4


### Query 2: Output per Line Produksi
**Deskripsi**: Menganalisis performa produksi di setiap lini perakitan beserta penanggung jawabnya dan persentase kontribusinya terhadap total output.

In [8]:
query_2 = """
SELECT
    l.line_name,
    l.supervisor,
    SUM(pf.output_qty) AS total_output,
    SUM(pf.target_qty) AS total_target,
    ROUND(SUM(pf.output_qty) * 100.0 / SUM(pf.target_qty), 1) AS achievement_pct,
    ROUND(SUM(pf.output_qty) * 100.0 / (SELECT SUM(output_qty) FROM production_fact), 1) AS output_share_pct
FROM production_fact pf
JOIN line_dim l ON pf.line_id = l.line_id
GROUP BY l.line_name, l.supervisor
ORDER BY total_output DESC;
"""
pd.read_sql_query(query_2, conn)

,line_name,supervisor,total_output,total_target,achievement_pct,output_share_pct
0,Line A,Budi Santoso,203533,227866,89.3,21.2
1,Line C,Ahmad Wijaya,201709,218199,92.4,21.0
2,Line B,Siti Rahayu,192078,211026,91.0,20.0
3,Line D,Dewi Kusuma,184814,202350,91.3,19.2
4,Line E,Rizky Pratama,179858,193187,93.1,18.7


### Query 3: Defect Rate per Shift
**Deskripsi**: Mengidentifikasi tingkat kecacatan produk (defect rate) di setiap shift kerja serta mengkategorikannya menjadi kategori High/Medium/Low.

In [9]:
query_3 = """
SELECT
    s.shift_name,
    s.start_time,
    s.end_time,
    SUM(pf.output_qty) AS total_output,
    SUM(pf.defect_qty) AS total_defect,
    ROUND(SUM(pf.defect_qty) * 100.0 / SUM(pf.output_qty), 2) AS defect_rate_pct,
    CASE
        WHEN SUM(pf.defect_qty) * 100.0 / SUM(pf.output_qty) > 4 THEN 'HIGH'
        WHEN SUM(pf.defect_qty) * 100.0 / SUM(pf.output_qty) > 3 THEN 'MEDIUM'
        ELSE 'LOW'
    END AS defect_category
FROM production_fact pf
JOIN shift_dim s ON pf.shift_id = s.shift_id
GROUP BY s.shift_name, s.start_time, s.end_time
ORDER BY defect_rate_pct DESC;
"""
pd.read_sql_query(query_3, conn)

,shift_name,start_time,end_time,total_output,total_defect,defect_rate_pct,defect_category
0,Shift Malam,22:00,06:00,292525,14869,5.08,HIGH
1,Shift Siang,14:00,22:00,326243,9517,2.92,LOW
2,Shift Pagi,06:00,14:00,343224,7570,2.21,LOW


### Query 4: Top 5 Mesin dengan Downtime Tertinggi
**Deskripsi**: Mencari 5 mesin manufaktur yang paling sering mengalami breakdown/downtime dan hubungannya dengan umur mesin.

In [10]:
query_4 = """
SELECT
    m.machine_name,
    m.machine_type,
    m.purchase_year,
    (2024 - m.purchase_year) AS machine_age,
    SUM(pf.downtime_minutes) AS total_downtime,
    ROUND(AVG(pf.downtime_minutes), 1) AS avg_downtime,
    COUNT(*) AS batch_count
FROM production_fact pf
JOIN machine_dim m ON pf.machine_id = m.machine_id
GROUP BY m.machine_name, m.machine_type, m.purchase_year
ORDER BY total_downtime DESC
LIMIT 5;
"""
pd.read_sql_query(query_4, conn)

,machine_name,machine_type,purchase_year,machine_age,total_downtime,avg_downtime,batch_count
0,WAVE-01,Wave Soldering,2017,7,60981.6,37.0,1646
1,SMT-01,SMT Placement,2018,6,58544.8,35.5,1648
2,ICT-01,Testing,2018,6,57102.5,34.7,1645
3,REFLOW-01,Reflow Oven,2016,8,57033.5,34.5,1653
4,AOI-01,Inspection,2019,5,39618.1,24.1,1647


### Query 5: Running Total Output Bulanan
**Deskripsi**: Menggunakan Window Function `SUM OVER` untuk menghitung akumulasi total output bulanan dan persentase kontribusinya secara kumulatif.

In [11]:
query_5 = """
SELECT
    month_num,
    monthly_output,
    SUM(monthly_output) OVER (ORDER BY month_num) AS running_total,
    ROUND(monthly_output * 100.0 / SUM(monthly_output) OVER (), 1) AS pct_of_total
FROM (
    SELECT
        CAST(strftime('%m', production_date) AS INTEGER) AS month_num,
        SUM(output_qty) AS monthly_output
    FROM production_fact
    GROUP BY CAST(strftime('%m', production_date) AS INTEGER)
) monthly_data
ORDER BY month_num;
"""
pd.read_sql_query(query_5, conn)

,month_num,monthly_output,running_total,pct_of_total
0,7,176714,176714,18.4
1,8,170244,346958,17.7
2,9,160505,507463,16.7
3,10,167325,674788,17.4
4,11,149804,824592,15.6
5,12,137400,961992,14.3


### Query 6: Ranking Lini Produksi Berdasarkan Output per Bulan
**Deskripsi**: Menggunakan `RANK()` dan `DENSE_RANK()` untuk mengurutkan performa lini produksi terbaik di setiap bulannya.

In [12]:
query_6 = """
SELECT
    CAST(strftime('%m', pf.production_date) AS INTEGER) AS month_num,
    l.line_name,
    SUM(pf.output_qty) AS monthly_output,
    RANK() OVER (
        PARTITION BY CAST(strftime('%m', pf.production_date) AS INTEGER)
        ORDER BY SUM(pf.output_qty) DESC
    ) AS output_rank,
    DENSE_RANK() OVER (
        PARTITION BY CAST(strftime('%m', pf.production_date) AS INTEGER)
        ORDER BY SUM(pf.output_qty) DESC
    ) AS dense_rank
FROM production_fact pf
JOIN line_dim l ON pf.line_id = l.line_id
GROUP BY CAST(strftime('%m', pf.production_date) AS INTEGER), l.line_name
ORDER BY month_num, output_rank;
"""
pd.read_sql_query(query_6, conn)

,month_num,line_name,monthly_output,output_rank,dense_rank
0,7,Line A,38359,1,1
1,7,Line C,36731,2,2
2,7,Line B,34980,3,3
3,7,Line D,33847,4,4
4,7,Line E,32797,5,5
5,8,Line A,36957,1,1
6,8,Line C,35340,2,2
7,8,Line B,33898,3,3
8,8,Line D,32473,4,4
9,8,Line E,31576,5,5


### Query 7: Moving Average Output 7 Hari
**Deskripsi**: Menggunakan window frame `ROWS BETWEEN` untuk menghitung rata-rata bergerak 7 hari guna meredam noise/fluktuasi harian.

In [13]:
query_7 = """
SELECT
    production_date,
    daily_output,
    ROUND(AVG(daily_output) OVER (
        ORDER BY production_date
        ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ), 0) AS moving_avg_7d
FROM (
    SELECT
        production_date,
        SUM(output_qty) AS daily_output
    FROM production_fact
    GROUP BY production_date
) daily_data
ORDER BY production_date;
"""
pd.read_sql_query(query_7, conn)

,production_date,daily_output,moving_avg_7d
0,2024-07-01,6155,6155.0
1,2024-07-02,6985,6570.0
2,2024-07-03,6833,6658.0
3,2024-07-04,6912,6721.0
4,2024-07-05,6914,6760.0
...,...,...,...
179,2024-12-27,5450,4502.0
180,2024-12-28,3053,4497.0
181,2024-12-29,1679,4503.0
182,2024-12-30,4709,4492.0


### Query 8: Achievement Rate per Line per Bulan
**Deskripsi**: Mengukur persentase pencapaian target produksi bulanan untuk setiap lini perakitan.

In [14]:
query_8 = """
SELECT
    l.line_name,
    CAST(strftime('%m', pf.production_date) AS INTEGER) AS month_num,
    SUM(pf.output_qty) AS total_output,
    SUM(pf.target_qty) AS total_target,
    ROUND(SUM(pf.output_qty) * 100.0 / SUM(pf.target_qty), 1) AS achievement_pct,
    CASE
        WHEN SUM(pf.output_qty) * 100.0 / SUM(pf.target_qty) >= 95 THEN 'ON TARGET'
        WHEN SUM(pf.output_qty) * 100.0 / SUM(pf.target_qty) >= 85 THEN 'BELOW TARGET'
        ELSE 'CRITICAL'
    END AS status
FROM production_fact pf
JOIN line_dim l ON pf.line_id = l.line_id
GROUP BY l.line_name, CAST(strftime('%m', pf.production_date) AS INTEGER)
ORDER BY l.line_name, month_num;
"""
pd.read_sql_query(query_8, conn)

,line_name,month_num,total_output,total_target,achievement_pct,status
0,Line A,7,38359,39002,98.4,ON TARGET
1,Line A,8,36957,38388,96.3,ON TARGET
2,Line A,9,34762,36498,95.2,ON TARGET
3,Line A,10,36123,39125,92.3,BELOW TARGET
4,Line A,11,32588,36891,88.3,BELOW TARGET
5,Line A,12,24744,37962,65.2,CRITICAL
6,Line B,7,34980,35987,97.2,ON TARGET
7,Line B,8,33898,35484,95.5,ON TARGET
8,Line B,9,31835,33769,94.3,BELOW TARGET
9,Line B,10,33043,36311,91.0,BELOW TARGET


### Query 9: Mesin dengan Downtime di Atas Rata-rata
**Deskripsi**: Menggunakan Subquery dan klausul `HAVING` untuk menyaring mesin-mesin yang memiliki rata-rata downtime melebihi rata-rata pabrik.

In [15]:
query_9 = """
SELECT
    m.machine_name,
    m.machine_type,
    (2024 - m.purchase_year) AS machine_age,
    ROUND(AVG(pf.downtime_minutes), 1) AS avg_downtime
FROM production_fact pf
JOIN machine_dim m ON pf.machine_id = m.machine_id
GROUP BY m.machine_name, m.machine_type, m.purchase_year
HAVING AVG(pf.downtime_minutes) > (
    SELECT AVG(downtime_minutes) FROM production_fact
)
ORDER BY avg_downtime DESC;
"""
pd.read_sql_query(query_9, conn)

,machine_name,machine_type,machine_age,avg_downtime
0,WAVE-01,Wave Soldering,7,37.0
1,SMT-01,SMT Placement,6,35.5
2,ICT-01,Testing,6,34.7
3,REFLOW-01,Reflow Oven,8,34.5
4,AOI-01,Inspection,5,24.1
5,REFLOW-02,Reflow Oven,4,23.0
6,PRESS-01,Press,5,22.9
7,WAVE-02,Wave Soldering,4,22.5


### Query 10: Perbandingan Output Bulan Ini vs Bulan Lalu (MoM Growth)
**Deskripsi**: Menggunakan `LAG()` untuk menghitung selisih dan persentase perubahan output produksi dari bulan ke bulan.

In [16]:
query_10 = """
SELECT
    month_num,
    monthly_output,
    LAG(monthly_output) OVER (ORDER BY month_num) AS prev_month_output,
    monthly_output - LAG(monthly_output) OVER (ORDER BY month_num) AS mom_change,
    ROUND(
        (monthly_output - LAG(monthly_output) OVER (ORDER BY month_num)) * 100.0
        / LAG(monthly_output) OVER (ORDER BY month_num), 1
    ) AS mom_change_pct
FROM (
    SELECT
        CAST(strftime('%m', production_date) AS INTEGER) AS month_num,
        SUM(output_qty) AS monthly_output
    FROM production_fact
    GROUP BY CAST(strftime('%m', production_date) AS INTEGER)
) monthly_data
ORDER BY month_num;
"""
pd.read_sql_query(query_10, conn)

,month_num,monthly_output,prev_month_output,mom_change,mom_change_pct
0,7,176714,NaN,NaN,NaN
1,8,170244,176714.0,-6470.0,-3.7
2,9,160505,170244.0,-9739.0,-5.7
3,10,167325,160505.0,6820.0,4.2
4,11,149804,167325.0,-17521.0,-10.5
5,12,137400,149804.0,-12404.0,-8.3


### Query 11: Defect Rate Trend Bulanan per Produk
**Deskripsi**: Melihat perkembangan defect rate setiap jenis produk (komponen) dari waktu ke waktu untuk QC evaluasi.

In [17]:
query_11 = """
SELECT
    CAST(strftime('%m', pf.production_date) AS INTEGER) AS month_num,
    p.product_name,
    SUM(pf.output_qty) AS total_output,
    SUM(pf.defect_qty) AS total_defect,
    ROUND(SUM(pf.defect_qty) * 100.0 / SUM(pf.output_qty), 2) AS defect_rate_pct
FROM production_fact pf
JOIN product_dim p ON pf.product_id = p.product_id
GROUP BY CAST(strftime('%m', pf.production_date) AS INTEGER), p.product_name
ORDER BY p.product_name, month_num;
"""
pd.read_sql_query(query_11, conn)

,month_num,product_name,total_output,total_defect,defect_rate_pct
0,7,Control Board,35852,2293,6.40
1,8,Control Board,34814,2272,6.53
2,9,Control Board,32609,2110,6.47
3,10,Control Board,34169,2267,6.63
4,11,Control Board,30363,2016,6.64
5,12,Control Board,27179,1900,6.99
6,7,LED Panel,33742,740,2.19
7,8,LED Panel,32762,713,2.18
8,9,LED Panel,30721,702,2.29
9,10,LED Panel,31895,727,2.28


### Query 12: Top Performing Line-Shift Combination
**Deskripsi**: Menggunakan CTE (Common Table Expression) untuk menganalisis produktivitas dan kualitas kombinasi Lini-Shift.

In [18]:
query_12 = """
WITH line_shift_perf AS (
    SELECT
        l.line_name,
        s.shift_name,
        SUM(pf.output_qty) AS total_output,
        ROUND(SUM(pf.defect_qty) * 100.0 / SUM(pf.output_qty), 2) AS defect_rate,
        ROUND(AVG(pf.downtime_minutes), 1) AS avg_downtime
    FROM production_fact pf
    JOIN line_dim l ON pf.line_id = l.line_id
    JOIN shift_dim s ON pf.shift_id = s.shift_id
    GROUP BY l.line_name, s.shift_name
)
SELECT
    line_name,
    shift_name,
    total_output,
    defect_rate,
    avg_downtime,
    RANK() OVER (ORDER BY total_output DESC) AS rank_by_output
FROM line_shift_perf
ORDER BY rank_by_output;
"""
pd.read_sql_query(query_12, conn)

,line_name,shift_name,total_output,defect_rate,avg_downtime,rank_by_output
0,Line A,Shift Pagi,72514,2.83,20.2,1
1,Line C,Shift Pagi,72012,2.55,23.4,2
2,Line A,Shift Siang,68765,3.64,20.5,3
3,Line C,Shift Siang,68615,3.36,23.9,4
4,Line B,Shift Pagi,68555,1.56,23.7,5
5,Line D,Shift Pagi,65961,1.54,22.8,6
6,Line B,Shift Siang,65199,2.10,25.2,7
7,Line E,Shift Pagi,64182,2.49,15.8,8
8,Line D,Shift Siang,62738,2.15,23.6,9
9,Line A,Shift Malam,62254,6.08,19.7,10


### Query 13: Identifikasi Hari dengan Output Anomali (di bawah 1.5 StdDev)
**Deskripsi**: Menemukan hari-hari di mana output produksi anjlok jauh di bawah rata-rata menggunakan fungsi kustom `STDDEV`.

In [19]:
query_13 = """
WITH daily_output AS (
    SELECT
        production_date,
        SUM(output_qty) AS total_daily_output
    FROM production_fact
    GROUP BY production_date
),
stats AS (
    SELECT
        AVG(total_daily_output) AS avg_daily,
        AVG(total_daily_output) - 1.5 * STDDEV(total_daily_output) AS lower_bound
    FROM daily_output
    -- Filter Hari Kerja (Senin s.d Jumat)
    WHERE CAST(strftime('%w', production_date) AS INTEGER) BETWEEN 1 AND 5
)
SELECT
    d.production_date,
    d.total_daily_output,
    ROUND(s.avg_daily, 0) AS avg_daily,
    ROUND(d.total_daily_output - s.avg_daily, 0) AS deviation
FROM daily_output d
CROSS JOIN stats s
WHERE d.total_daily_output < s.lower_bound
ORDER BY d.total_daily_output ASC
LIMIT 10;
"""
pd.read_sql_query(query_13, conn)

,production_date,total_daily_output,avg_daily,deviation
0,2024-12-22,1635,6213.0,-4578.0
1,2024-12-29,1679,6213.0,-4534.0
2,2024-12-01,1719,6213.0,-4494.0
3,2024-12-15,1720,6213.0,-4493.0
4,2024-12-08,1736,6213.0,-4477.0
5,2024-11-10,1821,6213.0,-4392.0
6,2024-11-17,1824,6213.0,-4389.0
7,2024-11-03,1848,6213.0,-4365.0
8,2024-11-24,1899,6213.0,-4314.0
9,2024-10-20,1911,6213.0,-4302.0


### Query 14: Ringkasan KPI Bulanan (Executive Dashboard)
**Deskripsi**: Menyusun rangkuman data eksekutif bulanan (Total Output, Target, Achievement %, Defect %, Downtime, Status warna).

In [20]:
query_14 = """
SELECT
    CAST(strftime('%m', production_date) AS INTEGER) AS month_num,
    SUM(output_qty) AS total_output,
    SUM(target_qty) AS total_target,
    ROUND(SUM(output_qty) * 100.0 / SUM(target_qty), 1) AS achievement_pct,
    SUM(defect_qty) AS total_defect,
    ROUND(SUM(defect_qty) * 100.0 / SUM(output_qty), 2) AS defect_rate_pct,
    ROUND(AVG(downtime_minutes), 1) AS avg_downtime,
    ROUND(SUM(downtime_minutes), 0) AS total_downtime,
    COUNT(DISTINCT production_date) AS working_days,
    CASE
        WHEN SUM(output_qty) * 100.0 / SUM(target_qty) >= 95 THEN 'GREEN'
        WHEN SUM(output_qty) * 100.0 / SUM(target_qty) >= 85 THEN 'YELLOW'
        ELSE 'RED'
    END AS status_flag
FROM production_fact
GROUP BY CAST(strftime('%m', production_date) AS INTEGER)
ORDER BY month_num;
"""
pd.read_sql_query(query_14, conn)

,month_num,total_output,total_target,achievement_pct,total_defect,defect_rate_pct,avg_downtime,total_downtime,working_days,status_flag
0,7,176714,180069,98.1,5801,3.28,21.2,88293.0,31,GREEN
1,8,170244,177233,96.1,5613,3.30,22.0,91410.0,31,GREEN
2,9,160505,168389,95.3,5257,3.28,20.9,84502.0,30,GREEN
3,10,167325,180980,92.5,5631,3.37,21.1,88033.0,31,YELLOW
4,11,149804,170635,87.8,4998,3.34,21.8,87724.0,30,YELLOW
5,12,137400,175322,78.4,4656,3.39,21.3,88927.0,31,RED


### Query 15: Analisis Dampak Downtime Terhadap Output per Line & Rekomendasi Pemeliharaan
**Deskripsi**: Menghitung rata-rata downtime mesin, rata-rata output, dan mengklasifikasikan mesin yang mendesak untuk diperbaiki.

In [21]:
query_15 = """
WITH machine_downtime AS (
    SELECT
        pf.line_id,
        l.line_name,
        m.machine_name,
        (2024 - m.purchase_year) AS machine_age,
        AVG(pf.downtime_minutes) AS avg_downtime,
        AVG(pf.output_qty) AS avg_output,
        COUNT(*) AS batch_count
    FROM production_fact pf
    JOIN line_dim l ON pf.line_id = l.line_id
    JOIN machine_dim m ON pf.machine_id = m.machine_id
    GROUP BY pf.line_id, l.line_name, m.machine_name, m.purchase_year
)
SELECT
    line_name,
    machine_name,
    machine_age,
    ROUND(avg_downtime, 1) AS avg_downtime_min,
    ROUND(avg_output, 0) AS avg_output_units,
    batch_count,
    CASE
        WHEN machine_age > 5 AND avg_downtime > 30 THEN 'PRIORITAS TINGGI - Perlu preventive maintenance segera'
        WHEN machine_age > 3 AND avg_downtime > 20 THEN 'PRIORITAS SEDANG - Jadwalkan maintenance rutin'
        ELSE 'NORMAL - Lanjutkan monitoring'
    END AS maintenance_recommendation
FROM machine_downtime
ORDER BY avg_downtime DESC;
"""
pd.read_sql_query(query_15, conn)

,line_name,machine_name,machine_age,avg_downtime_min,avg_output_units,batch_count,maintenance_recommendation
0,Line B,WAVE-01,7,37.0,38.0,1646,PRIORITAS TINGGI - Perlu preventive maintenanc...
1,Line A,SMT-01,6,35.5,40.0,1648,PRIORITAS TINGGI - Perlu preventive maintenanc...
2,Line C,ICT-01,6,34.7,40.0,1645,PRIORITAS TINGGI - Perlu preventive maintenanc...
3,Line D,REFLOW-01,8,34.5,36.0,1653,PRIORITAS TINGGI - Perlu preventive maintenanc...
4,Line C,AOI-01,5,24.1,41.0,1647,PRIORITAS SEDANG - Jadwalkan maintenance rutin
5,Line D,REFLOW-02,4,23.0,38.0,1646,PRIORITAS SEDANG - Jadwalkan maintenance rutin
6,Line E,PRESS-01,5,22.9,36.0,1649,PRIORITAS SEDANG - Jadwalkan maintenance rutin
7,Line B,WAVE-02,4,22.5,39.0,1642,PRIORITAS SEDANG - Jadwalkan maintenance rutin
8,Line E,REFLOW-03,0,12.7,36.0,1649,NORMAL - Lanjutkan monitoring
9,Line A,SMT-02,3,12.6,42.0,1648,NORMAL - Lanjutkan monitoring


## 7. Membaca Database Views

### View: Prioritas Pemeliharaan Mesin

In [22]:
pd.read_sql_query("SELECT * FROM v_machine_maintenance_priority WHERE maintenance_recommendation LIKE 'PRIORITAS%'", conn)

,line_name,machine_name,machine_type,machine_age,avg_downtime_min,avg_output_units,batch_count,maintenance_recommendation
0,Line A,SMT-01,SMT Placement,6,35.5,40.0,1648,PRIORITAS TINGGI - Perlu preventive maintenanc...
1,Line B,WAVE-01,Wave Soldering,7,37.0,38.0,1646,PRIORITAS TINGGI - Perlu preventive maintenanc...
2,Line B,WAVE-02,Wave Soldering,4,22.5,39.0,1642,PRIORITAS SEDANG - Jadwalkan maintenance rutin
3,Line C,AOI-01,Inspection,5,24.1,41.0,1647,PRIORITAS SEDANG - Jadwalkan maintenance rutin
4,Line C,ICT-01,Testing,6,34.7,40.0,1645,PRIORITAS TINGGI - Perlu preventive maintenanc...
5,Line D,REFLOW-01,Reflow Oven,8,34.5,36.0,1653,PRIORITAS TINGGI - Perlu preventive maintenanc...
6,Line D,REFLOW-02,Reflow Oven,4,23.0,38.0,1646,PRIORITAS SEDANG - Jadwalkan maintenance rutin
7,Line E,PRESS-01,Press,5,22.9,36.0,1649,PRIORITAS SEDANG - Jadwalkan maintenance rutin


### View: Analisis Defect & Kerugian Keuangan per Produk

In [23]:
pd.read_sql_query("SELECT * FROM v_product_defect_summary ORDER BY total_defect_cost_idr DESC", conn)

,product_name,category,total_output,total_defect,defect_rate_pct,cost_per_unit_idr,total_defect_cost_idr
0,Control Board,Controller,194986,12858,6.59,120000,1.542960e+09
1,Power Supply Unit,Power,193243,3837,1.99,78000,2.992860e+08
2,Sensor Module,Component,193856,6171,3.18,45000,2.776950e+08
3,LED Panel,Display,185397,4162,2.24,35000,1.456700e+08
4,PCB Assembly,Assembly,194510,4928,2.53,25000,1.232000e+08


In [24]:
# Tutup Koneksi Database
conn.close()
print("Koneksi database berhasil ditutup secara aman.")

Koneksi database berhasil ditutup secara aman.
